**Project Code:** P001-MVB
**Stage:** 04_MVB_Feature-Engineering
**Notebook Version:** v1.0
**Author:** Abubakar Amidu
**Programme:** 3MTT DeepTech Cohort 2 — DS/ML Mentorship
**Last Updated:** 25 July 2026

---

# MVB-04 — Feature Engineering

**Project:** P001-MVB — Explainable and Responsible AI for Differentiating Bacterial and Viral Meningitis: A Clinical Decision Support System

## Purpose

Resolve the open feature-level decisions identified in Stage 03, encode categorical variables, and produce the final modeling-ready feature sets.

## Inputs

- `MVB-02-cleaned-data/MVB-02-meningitis-cleaned.csv` (Stage 02 output, unchanged since)

## Outputs

- Feature Set A (Full): `MVB-04-results/MVB-04-feature-set-A-full.csv`
- Feature Set B (Restricted): `MVB-04-results/MVB-04-feature-set-B-restricted.csv`
- Feature engineering decision log: `MVB-04-results/MVB-04-feature-engineering-log.csv`
- This notebook
- Updated documentation

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Set working directory to Stage 04 folder
import os
os.chdir('/content/drive/MyDrive/P001-MVB/04_MVB_Feature-Engineering')

In [ ]:
# Ensure output folder exists
os.makedirs("MVB-04-results", exist_ok=True)

In [ ]:
# Imports
import pandas as pd
import numpy as np

# Reproducibility standard, consistent with Stage 03
np.random.seed(42)

In [ ]:

# Load the cleaned dataset from Stage 02 (unchanged since Stage 03)
df = pd.read_csv("../02_MVB_Data-Preparation/MVB-02-cleaned-data/MVB-02-meningitis-cleaned.csv")
print("Shape loaded:", df.shape)

Shape loaded: (1133, 14)


In [ ]:
# Confirm the data matches Stage 02/03's documented output before proceeding
assert df.shape == (1133, 14)
assert df["Diagnosis"].isin(["Bacterial","Viral"]).all()
assert df["Gender"].isin(["Female","Male"]).all()
assert df["Pathogen_Present"].isin(["Yes","No"]).all()
print("Integrity check passed: shape and category values confirmed.")

Integrity check passed: shape and category values confirmed.


## Decisions

### 1. `Pathogen_Present` — Near-Perfect Proxy Feature

Stage 03 found `Pathogen_Present` matches `Diagnosis` in 94% of cases — far beyond a genuine diagnostic relationship, more consistent with a near-proxy for the label itself. Including it risks producing a model that reproduces the label rather than learning from clinically meaningful CSF/blood markers.

**Decision:** Rather than simply including or excluding it, two parallel feature sets are produced:
- **Feature Set A (Full):** includes `Pathogen_Present`
- **Feature Set B (Restricted):** excludes `Pathogen_Present`

Both will be carried forward to Stage 05 as separate model variants for comparison. This turns a data-quality caveat into an explicit Responsible AI discussion point about proxy features and what a model is actually learning.

### 2. Multicollinearity

Stage 03 identified moderate-to-strong correlations among several features (Hemoglobin–Platelets: 0.77, WBC_Blood_Count–Platelets: -0.74, WBC_Count–Protein_Level: 0.70).

**Decision:** No features are dropped for correlation reasons. Random Forest and XGBoost (per the proposal's candidate algorithms) handle multicollinearity natively; only Logistic Regression is sensitive to it. Dropping features risks losing genuine diagnostic signal. These correlations are documented here as context for interpreting Stage 06 evaluation results, particularly for the Logistic Regression variant.

### 3. CSF-to-Blood Glucose Ratio — Evaluated, Not Engineered

The proposal identified a CSF-to-blood glucose ratio as an example of clinically meaningful feature engineering (a real diagnostic tool — a ratio below ~0.4–0.5 is a stronger bacterial-meningitis indicator than either value alone).

**Decision:** This feature is not engineered. The dataset contains no paired blood/serum glucose column — only CSF `Glucose_Level` exists — so the ratio as clinically defined cannot be computed. Fabricating an approximate substitute using an unrelated marker would introduce a feature with no genuine clinical basis, so this was declined rather than forced.

### 4. Encoding

All categorical variables in this dataset are binary, so label encoding (0/1) is used — equivalent to one-hot encoding for two categories, but simpler and keeps feature count lower. Original column names are retained after encoding (rather than appending `_Encoded`) for readability in downstream stages.

In [ ]:

# Encode target and binary categorical variables — original column names retained for readability
df_encoded = df.copy()
df_encoded["Diagnosis"] = df_encoded["Diagnosis"].map({"Viral": 0, "Bacterial": 1})
df_encoded["Gender"] = df_encoded["Gender"].map({"Female": 0, "Male": 1})
df_encoded["Pathogen_Present"] = df_encoded["Pathogen_Present"].map({"No": 0, "Yes": 1})

# Verify no nulls were introduced by the mapping
assert not df_encoded.isnull().any().any()
print("Encoding verified: no nulls introduced.")
print(df_encoded[["Diagnosis","Gender","Pathogen_Present"]].head())

Encoding verified: no nulls introduced.
   Diagnosis  Gender  Pathogen_Present
0          1       0                 1
1          1       1                 0
2          0       1                 0
3          1       0                 1
4          0       0                 0


**Target encoding rationale:** Bacterial = 1 (positive class), Viral = 0. This aligns with the proposal's priority on recall/sensitivity for bacterial meningitis — the positive class is the one where missed cases carry the greatest clinical risk.

## Construct Final Feature Sets

In [ ]:
# Feature Set A (Full) — includes Pathogen_Present
feature_set_a_cols = ["Age", "Gender", "WBC_Count", "Protein_Level", "Glucose_Level",
                       "Pathogen_Present", "Hemoglobin", "WBC_Blood_Count", "Platelets", "CRP_Level"]

# Feature Set B (Restricted) — excludes Pathogen_Present
feature_set_b_cols = ["Age", "Gender", "WBC_Count", "Protein_Level", "Glucose_Level",
                       "Hemoglobin", "WBC_Blood_Count", "Platelets", "CRP_Level"]

df_feature_set_a = df_encoded[feature_set_a_cols + ["Diagnosis"]].copy()
df_feature_set_b = df_encoded[feature_set_b_cols + ["Diagnosis"]].copy()

print("Feature Set A (Full) shape:", df_feature_set_a.shape)
print("Feature Set A columns:", feature_set_a_cols)
print()
print("Feature Set B (Restricted) shape:", df_feature_set_b.shape)
print("Feature Set B columns:", feature_set_b_cols)

Feature Set A (Full) shape: (1133, 11)
Feature Set A columns: ['Age', 'Gender', 'WBC_Count', 'Protein_Level', 'Glucose_Level', 'Pathogen_Present', 'Hemoglobin', 'WBC_Blood_Count', 'Platelets', 'CRP_Level']

Feature Set B (Restricted) shape: (1133, 10)
Feature Set B columns: ['Age', 'Gender', 'WBC_Count', 'Protein_Level', 'Glucose_Level', 'Hemoglobin', 'WBC_Blood_Count', 'Platelets', 'CRP_Level']


In [ ]:

# Save both feature sets
df_feature_set_a.to_csv("MVB-04-results/MVB-04-feature-set-A-full.csv", index=False)
df_feature_set_b.to_csv("MVB-04-results/MVB-04-feature-set-B-restricted.csv", index=False)
print("Both feature sets saved.")

Both feature sets saved.


## Export Feature Engineering Decision Log

In [ ]:

# Save a summary log of every decision made in this stage — reusable for documentation and the final report
from datetime import datetime

feature_log = pd.DataFrame([
    {"Decision":"Pathogen_Present","Action":"Two feature sets created","Rationale":"Near-perfect proxy (94% match) - avoid trivial model; supports Responsible AI comparison"},
    {"Decision":"Multicollinearity","Action":"Retained, documented only","Rationale":"Tree-based models handle natively; dropping risks losing signal"},
    {"Decision":"CSF-to-blood glucose ratio","Action":"Not engineered","Rationale":"No paired blood glucose column exists in dataset"},
    {"Decision":"Categorical encoding","Action":"Label encoding (0/1), original column names retained","Rationale":"Binary categories - equivalent to one-hot, simpler, cleaner for downstream stages"},
    {"Decision":"Target encoding","Action":"Bacterial=1, Viral=0","Rationale":"Aligns with proposal priority on Bacterial recall/sensitivity"},
])
feature_log["Decision_Date"] = datetime.today().date()
feature_log.to_csv("MVB-04-results/MVB-04-feature-engineering-log.csv", index=False)
print(feature_log)

                     Decision  \
0            Pathogen_Present   
1           Multicollinearity   
2  CSF-to-blood glucose ratio   
3        Categorical encoding   
4             Target encoding   

                                              Action  \
0                           Two feature sets created   
1                          Retained, documented only   
2                                     Not engineered   
3  Label encoding (0/1), original column names re...   
4                               Bacterial=1, Viral=0   

                                           Rationale Decision_Date  
0  Near-perfect proxy (94% match) - avoid trivial...    2026-07-25  
1  Tree-based models handle natively; dropping ri...    2026-07-25  
2   No paired blood glucose column exists in dataset    2026-07-25  
3  Binary categories - equivalent to one-hot, sim...    2026-07-25  
4  Aligns with proposal priority on Bacterial rec...    2026-07-25  


## Summary

- `Pathogen_Present`: not simply included or excluded — two parallel feature sets (A/Full, B/Restricted) produced for comparative modeling in Stage 05.
- Multicollinearity: documented, not acted upon at this stage — deferred to model choice and Stage 06 interpretation.
- CSF-to-blood glucose ratio: evaluated and declined — infeasible given available columns.
- Encoding: label encoding for all binary categoricals, original column names retained; Bacterial=1 as positive class, aligned with the project's recall priority.
- Feature Set A: 10 features + target, 1,133 rows.
- Feature Set B: 9 features + target, 1,133 rows.

---

## Stage Status

**Stage:** Completed

**Primary Outputs Produced:**
✓ Feature Set A — Full (MVB-04-feature-set-A-full.csv)
✓ Feature Set B — Restricted (MVB-04-feature-set-B-restricted.csv)
✓ Feature engineering decision log (MVB-04-feature-engineering-log.csv)
✓ Feature engineering notebook
✓ Updated documentation

**Input for Next Stage:**
`MVB-04-results/MVB-04-feature-set-A-full.csv` and `MVB-04-results/MVB-04-feature-set-B-restricted.csv`

**Next Stage:**
05_MVB_Model-Training — Train and compare Logistic Regression, Random Forest, and XGBoost on both feature sets.

In [12]:
# Notebook completion timestamp and environment info
from datetime import datetime
import platform

print("="*60)
print("Stage 04 completed successfully")
print("Completion time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("="*60)

Stage 04 completed successfully
Completion time: 2026-07-25 20:33:09
Python: 3.12.13
Pandas: 2.2.2
